# 01. Формирование raw-выборки для HDBSCAN

Этот ноутбук **только создает директорию с выборкой** из `data/raw/wb_feedbacks_full`.

Логика:

```text
идем по raw parquet-файлам подряд
из каждого файла берем 100 случайных строк
каждые 1000 строк сохраняем в отдельный sample_part_*.parquet
цель: 200 файлов × 1000 строк = 200 000 отзывов
```

В этой версии чтение каждого parquet-файла запускается **в отдельном subprocess**. Если файл обрабатывается дольше `20` секунд, subprocess принудительно убивается, файл пропускается, и цикл идет дальше.


In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
!pip -q install pyarrow tqdm

In [9]:
from pathlib import Path
import os
import sys
import json
import time
import shutil
import subprocess
import pandas as pd
from tqdm.auto import tqdm

# =========================
# Основные пути
# =========================

BASE_DIR = Path('/content/drive/MyDrive/MLops_project/project')
DATA_DIR = BASE_DIR / 'data'

RAW_DIR = DATA_DIR / 'raw' / 'wb_feedbacks_full'

# Новая директория внутри data
SAMPLE_DIR = DATA_DIR / 'hdbscan_raw_sample_200_files_1000_each'
TMP_DIR = SAMPLE_DIR / '_tmp_one_file_samples'

MANIFEST_PATH = SAMPLE_DIR / 'sample_manifest.csv'
META_PATH = SAMPLE_DIR / 'sample_meta.json'

# =========================
# Настройки выборки
# =========================

SAMPLES_PER_RAW_FILE = 100
ROWS_PER_OUTPUT_FILE = 1000
TARGET_OUTPUT_FILES = 200
TARGET_TOTAL_ROWS = ROWS_PER_OUTPUT_FILE * TARGET_OUTPUT_FILES

RANDOM_SEED = 42
READ_TIMEOUT_SECONDS = 20

# Если True — удалить старую директорию SAMPLE_DIR и собрать заново.
# Сейчас лучше оставить True, потому что у тебя уже есть неполная директория с 1 файлом.
OVERWRITE_SAMPLE_DIR = True

# Если True — сохранять только текстовый столбец + служебные поля.
# Если False — сохранять все столбцы исходного parquet.
KEEP_ONLY_TEXT_AND_META = False

# Возможные названия текстового столбца
TEXT_COL_CANDIDATES = ['text', 'review', 'review_text', 'comment', 'content']

print('RAW_DIR:', RAW_DIR)
print('SAMPLE_DIR:', SAMPLE_DIR)
print('TARGET_TOTAL_ROWS:', TARGET_TOTAL_ROWS)
print('READ_TIMEOUT_SECONDS:', READ_TIMEOUT_SECONDS)

RAW_DIR: /content/drive/MyDrive/MLops_project/project/data/raw/wb_feedbacks_full
SAMPLE_DIR: /content/drive/MyDrive/MLops_project/project/data/hdbscan_raw_sample_200_files_1000_each
TARGET_TOTAL_ROWS: 200000
READ_TIMEOUT_SECONDS: 20


In [10]:
# =========================
# Подготовка директорий
# =========================

if not RAW_DIR.exists():
    raise FileNotFoundError(f'Не найдена raw-директория: {RAW_DIR}')

raw_files = sorted(RAW_DIR.glob('*.parquet'))

print(f'Найдено raw parquet-файлов: {len(raw_files)}')

if len(raw_files) == 0:
    raise RuntimeError(f'В директории {RAW_DIR} нет parquet-файлов.')

if OVERWRITE_SAMPLE_DIR and SAMPLE_DIR.exists():
    print(f'Удаляю старую директорию: {SAMPLE_DIR}')
    shutil.rmtree(SAMPLE_DIR)
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
TMP_DIR.mkdir(parents=True, exist_ok=True)

existing_sample_files = sorted(SAMPLE_DIR.glob('sample_part_*.parquet'))
print(f'Уже существующих sample-файлов: {len(existing_sample_files)}')

if existing_sample_files and not OVERWRITE_SAMPLE_DIR:
    print('ВНИМАНИЕ: sample-файлы уже существуют. Чтобы пересобрать выборку, поставь OVERWRITE_SAMPLE_DIR = True.')

Найдено raw parquet-файлов: 1903
Удаляю старую директорию: /content/drive/MyDrive/MLops_project/project/data/hdbscan_raw_sample_200_files_1000_each
Уже существующих sample-файлов: 0


In [11]:
# =========================
# Worker-скрипт для обработки одного parquet-файла
# =========================
# Он будет запускаться отдельным Python-процессом через subprocess.run(..., timeout=20).
# Это надежнее, чем multiprocessing внутри ноутбука: если pd.read_parquet зависнет,
# весь subprocess будет убит по таймауту.

WORKER_SCRIPT_PATH = SAMPLE_DIR / '_sample_one_parquet_worker.py'

worker_code = r'''
import argparse
from pathlib import Path
import pandas as pd

TEXT_COL_CANDIDATES = ['text', 'review', 'review_text', 'comment', 'content']

def detect_text_col(columns):
    for c in TEXT_COL_CANDIDATES:
        if c in columns:
            return c
    # fallback: первый object/string столбец будет выбран уже после чтения
    return None

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--input', required=True)
    parser.add_argument('--output', required=True)
    parser.add_argument('--n', type=int, required=True)
    parser.add_argument('--seed', type=int, required=True)
    parser.add_argument('--keep_only_text_and_meta', type=int, default=0)
    args = parser.parse_args()

    input_path = Path(args.input)
    output_path = Path(args.output)

    df = pd.read_parquet(input_path)

    if len(df) == 0:
        raise RuntimeError('empty parquet file')

    text_col = detect_text_col(df.columns)

    if text_col is None:
        # fallback: пробуем найти первый строковый столбец
        string_cols = []
        for col in df.columns:
            try:
                if pd.api.types.is_string_dtype(df[col]) or df[col].dtype == 'object':
                    string_cols.append(col)
            except Exception:
                pass

        if not string_cols:
            raise RuntimeError(f'no text-like column found; columns={list(df.columns)}')

        text_col = string_cols[0]

    # Удаляем пустые тексты только для выборки, чтобы не тянуть мусор
    text_series = df[text_col].astype(str)
    mask = text_series.str.strip().ne('') & text_series.str.lower().ne('nan')
    df = df.loc[mask].copy()

    if len(df) == 0:
        raise RuntimeError('no non-empty text rows after filtering')

    # сохраняем исходный индекс строки до reset_index
    df['source_row_idx'] = df.index
    df['source_file'] = input_path.name
    df['source_text_col'] = text_col

    take_n = min(args.n, len(df))
    sample_df = df.sample(n=take_n, random_state=args.seed)

    if args.keep_only_text_and_meta:
        sample_df = sample_df[[text_col, 'source_row_idx', 'source_file', 'source_text_col']].copy()
        if text_col != 'text':
            sample_df = sample_df.rename(columns={text_col: 'text'})

    sample_df = sample_df.reset_index(drop=True)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    sample_df.to_parquet(output_path, index=False)

    print(f'ok rows={len(sample_df)} text_col={text_col}')

if __name__ == '__main__':
    main()
'''

WORKER_SCRIPT_PATH.write_text(worker_code, encoding='utf-8')
print('Worker script saved:', WORKER_SCRIPT_PATH)

Worker script saved: /content/drive/MyDrive/MLops_project/project/data/hdbscan_raw_sample_200_files_1000_each/_sample_one_parquet_worker.py


In [12]:
# =========================
# Генерация sample-файлов
# =========================

existing_sample_files = sorted(SAMPLE_DIR.glob('sample_part_*.parquet'))

if len(existing_sample_files) >= TARGET_OUTPUT_FILES and not OVERWRITE_SAMPLE_DIR:
    print(f'Генерация пропущена: уже есть {len(existing_sample_files)} sample-файлов.')
else:
    # Если директория была неполной, а OVERWRITE_SAMPLE_DIR=False, лучше явно остановиться.
    # Так мы не смешаем старую неполную выборку с новой.
    if existing_sample_files and not OVERWRITE_SAMPLE_DIR:
        raise RuntimeError(
            f'Найдены неполные sample-файлы: {len(existing_sample_files)}. '
            f'Поставь OVERWRITE_SAMPLE_DIR=True, чтобы пересобрать выборку заново.'
        )

    buffer_parts = []
    buffer_rows = 0
    saved_files = 0
    sampled_rows_total = 0

    processed_raw_files = 0
    skipped_raw_files = 0
    timeout_raw_files = 0
    error_raw_files = 0

    manifest_rows = []

    # Создаем/перезаписываем manifest сразу, чтобы прогресс был виден даже при остановке
    if MANIFEST_PATH.exists():
        MANIFEST_PATH.unlink()

    pbar = tqdm(raw_files, desc='Sampling raw parquet files')

    for raw_idx, raw_path in enumerate(pbar):
        if saved_files >= TARGET_OUTPUT_FILES:
            break

        processed_raw_files += 1
        t0 = time.time()

        tmp_out = TMP_DIR / f'one_file_sample_{raw_idx:06d}.parquet'
        if tmp_out.exists():
            tmp_out.unlink()

        cmd = [
            sys.executable,
            str(WORKER_SCRIPT_PATH),
            '--input', str(raw_path),
            '--output', str(tmp_out),
            '--n', str(SAMPLES_PER_RAW_FILE),
            '--seed', str(RANDOM_SEED + raw_idx),
            '--keep_only_text_and_meta', str(int(KEEP_ONLY_TEXT_AND_META)),
        ]

        status = 'unknown'
        rows_taken = 0
        error_msg = ''

        try:
            completed = subprocess.run(
                cmd,
                timeout=READ_TIMEOUT_SECONDS,
                capture_output=True,
                text=True,
            )

            elapsed = time.time() - t0

            if completed.returncode != 0:
                status = 'error'
                error_raw_files += 1
                skipped_raw_files += 1
                error_msg = (completed.stderr or completed.stdout or '')[-1000:]
            else:
                if not tmp_out.exists():
                    status = 'error_no_output'
                    error_raw_files += 1
                    skipped_raw_files += 1
                    error_msg = 'worker finished with code 0 but output file was not created'
                else:
                    part_df = pd.read_parquet(tmp_out)
                    rows_taken = len(part_df)

                    if rows_taken == 0:
                        status = 'empty_sample'
                        skipped_raw_files += 1
                    else:
                        status = 'ok'
                        buffer_parts.append(part_df)
                        buffer_rows += rows_taken
                        sampled_rows_total += rows_taken

                        # Каждые 1000 строк сохраняем sample_part
                        while buffer_rows >= ROWS_PER_OUTPUT_FILE and saved_files < TARGET_OUTPUT_FILES:
                            current = pd.concat(buffer_parts, ignore_index=True)

                            out_df = current.iloc[:ROWS_PER_OUTPUT_FILE].copy()
                            rest_df = current.iloc[ROWS_PER_OUTPUT_FILE:].copy()

                            out_df['sample_part_id'] = saved_files

                            out_path = SAMPLE_DIR / f'sample_part_{saved_files:05d}.parquet'
                            out_df.to_parquet(out_path, index=False)

                            saved_files += 1

                            if len(rest_df) > 0:
                                buffer_parts = [rest_df]
                                buffer_rows = len(rest_df)
                            else:
                                buffer_parts = []
                                buffer_rows = 0

        except subprocess.TimeoutExpired:
            elapsed = time.time() - t0
            status = 'timeout'
            skipped_raw_files += 1
            timeout_raw_files += 1
            error_msg = f'timeout after {READ_TIMEOUT_SECONDS} seconds'

        except Exception as e:
            elapsed = time.time() - t0
            status = 'main_error'
            skipped_raw_files += 1
            error_raw_files += 1
            error_msg = repr(e)

        manifest_rows.append({
            'raw_idx': raw_idx,
            'raw_file': raw_path.name,
            'status': status,
            'rows_taken': rows_taken,
            'elapsed_sec': round(elapsed, 3),
            'saved_files_after': saved_files,
            'buffer_rows_after': buffer_rows,
            'error': error_msg,
        })

        # Сохраняем manifest после каждого raw-файла
        MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
        pd.DataFrame(manifest_rows).to_csv(MANIFEST_PATH, index=False)

        pbar.set_postfix({
            'saved_files': saved_files,
            'buffer_rows': buffer_rows,
            'sampled_rows': sampled_rows_total,
            'skipped': skipped_raw_files,
            'timeouts': timeout_raw_files,
            'last': status,
        })

    meta = {
        'raw_dir': str(RAW_DIR),
        'sample_dir': str(SAMPLE_DIR),
        'samples_per_raw_file': SAMPLES_PER_RAW_FILE,
        'rows_per_output_file': ROWS_PER_OUTPUT_FILE,
        'target_output_files': TARGET_OUTPUT_FILES,
        'target_total_rows': TARGET_TOTAL_ROWS,
        'random_seed': RANDOM_SEED,
        'read_timeout_seconds': READ_TIMEOUT_SECONDS,
        'keep_only_text_and_meta': KEEP_ONLY_TEXT_AND_META,
        'processed_raw_files': processed_raw_files,
        'skipped_raw_files': skipped_raw_files,
        'timeout_raw_files': timeout_raw_files,
        'error_raw_files': error_raw_files,
        'saved_output_files': saved_files,
        'sampled_rows_total_before_packaging': sampled_rows_total,
        'final_full_rows_saved': saved_files * ROWS_PER_OUTPUT_FILE,
        'manifest_path': str(MANIFEST_PATH),
    }

    META_PATH.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding='utf-8')

    print('Готово')
    print(f'Сохранено sample-файлов: {saved_files}')
    print(f'Сохранено строк в полных sample-файлах: {saved_files * ROWS_PER_OUTPUT_FILE}')
    print(f'Пропущено raw-файлов всего: {skipped_raw_files}')
    print(f'Из них по timeout: {timeout_raw_files}')
    print(f'Из них по error: {error_raw_files}')
    print(f'Manifest: {MANIFEST_PATH}')
    print(f'Meta: {META_PATH}')

    if saved_files < TARGET_OUTPUT_FILES:
        print('ВНИМАНИЕ: не удалось получить все 200 файлов. Возможно, raw-файлов меньше, чем нужно, или много файлов было пропущено.')

Sampling raw parquet files:   0%|          | 0/1903 [00:00<?, ?it/s]

Готово
Сохранено sample-файлов: 190
Сохранено строк в полных sample-файлах: 190000
Пропущено raw-файлов всего: 0
Из них по timeout: 0
Из них по error: 0
Manifest: /content/drive/MyDrive/MLops_project/project/data/hdbscan_raw_sample_200_files_1000_each/sample_manifest.csv
Meta: /content/drive/MyDrive/MLops_project/project/data/hdbscan_raw_sample_200_files_1000_each/sample_meta.json
ВНИМАНИЕ: не удалось получить все 200 файлов. Возможно, raw-файлов меньше, чем нужно, или много файлов было пропущено.


In [13]:
# =========================
# Проверка результата
# =========================

sample_files = sorted(SAMPLE_DIR.glob('sample_part_*.parquet'))
print(f'Итоговых sample-файлов: {len(sample_files)}')

if sample_files:
    sizes = []
    for p in tqdm(sample_files, desc='Checking sample files'):
        sizes.append(len(pd.read_parquet(p)))

    print('Минимальный размер файла:', min(sizes))
    print('Максимальный размер файла:', max(sizes))
    print('Всего строк:', sum(sizes))

    display(pd.DataFrame({
        'file': [p.name for p in sample_files[:10]],
        'rows': sizes[:10],
    }))

    example_df = pd.read_parquet(sample_files[0])
    print('Колонки первого файла:')
    print(list(example_df.columns))
    display(example_df.head(3))

if MANIFEST_PATH.exists():
    manifest_df = pd.read_csv(MANIFEST_PATH)
    print('Статусы обработки raw-файлов:')
    display(manifest_df['status'].value_counts().reset_index().rename(columns={'index': 'status', 'status': 'count'}))
    display(manifest_df.tail(10))

Итоговых sample-файлов: 190


Checking sample files:   0%|          | 0/190 [00:00<?, ?it/s]

Минимальный размер файла: 1000
Максимальный размер файла: 1000
Всего строк: 190000


,file,rows
0,sample_part_00000.parquet,1000
1,sample_part_00001.parquet,1000
2,sample_part_00002.parquet,1000
3,sample_part_00003.parquet,1000
4,sample_part_00004.parquet,1000
5,sample_part_00005.parquet,1000
6,sample_part_00006.parquet,1000
7,sample_part_00007.parquet,1000
8,sample_part_00008.parquet,1000
9,sample_part_00009.parquet,1000


Колонки первого файла:
['nmId', 'productValuation', 'color', 'text', 'answer', 'source_row_idx', 'source_file', 'source_text_col', 'sample_part_id']


,nmId,productValuation,color,text,answer,source_row_idx,source_file,source_text_col,sample_part_id
0,0,5,темно-серый,качество хорошее. цвет темно серый. Очень прог...,,75721,raw_part_00000.parquet,text,0
1,0,5,черный,Классная лампа. Не хлипкая. Очень удобная. Бра...,Добрый день! благодарим за выбор нашей продукц...,80184,raw_part_00000.parquet,text,0
2,0,4,черный,На узкую ногу,Здравствуйте! Спасибо за отзыв. Мы очень сожал...,19864,raw_part_00000.parquet,text,0


Статусы обработки raw-файлов:


,count,count
0,ok,1903


,raw_idx,raw_file,status,rows_taken,elapsed_sec,saved_files_after,buffer_rows_after,error
1893,1893,raw_part_01893.parquet,ok,100,2.338,189,400,NaN
1894,1894,raw_part_01894.parquet,ok,100,2.578,189,500,NaN
1895,1895,raw_part_01895.parquet,ok,100,3.017,189,600,NaN
1896,1896,raw_part_01896.parquet,ok,100,3.796,189,700,NaN
1897,1897,raw_part_01897.parquet,ok,100,2.428,189,800,NaN
1898,1898,raw_part_01898.parquet,ok,100,2.862,189,900,NaN
1899,1899,raw_part_01899.parquet,ok,100,2.435,190,0,NaN
1900,1900,raw_part_01900.parquet,ok,100,3.318,190,100,NaN
1901,1901,raw_part_01901.parquet,ok,100,2.674,190,200,NaN
1902,1902,raw_part_01902.parquet,ok,100,2.440,190,300,NaN
